In [1]:
# Importing Libraries
import numpy as np
import pandas as pd

# Load the CSV files into DataFrames
customers_df = pd.read_csv("https://mentorskool-platform-uploads.s3.ap-south-1.amazonaws.com/documents/9eab6c04-532b-40d3-9d5e-2438ec366ee0_83d04ac6-cb74-4a96-a06a-e0d5442aa126_Dataset%20-%20Ecommerce.xlsx%20-%20customers(1).csv")
orders_df = pd.read_csv("https://mentorskool-platform-uploads.s3.ap-south-1.amazonaws.com/documents/ae936127-9b09-4c93-912e-1d6a862b5f82_83d04ac6-cb74-4a96-a06a-e0d5442aa126_Dataset%20-%20Ecommerce.xlsx%20-%20orders.csv")
transactions_df = pd.read_csv("https://mentorskool-platform-uploads.s3.ap-south-1.amazonaws.com/documents/dd2b3166-8bd1-4410-b2a2-a839345fb9e2_83d04ac6-cb74-4a96-a06a-e0d5442aa126_Dataset%20-%20Ecommerce.xlsx%20-%20transactions.csv")
returns_df = pd.read_csv("https://mentorskool-platform-uploads.s3.ap-south-1.amazonaws.com/documents/78adf709-576b-4d70-9ce8-418aca84b0ef_83d04ac6-cb74-4a96-a06a-e0d5442aa126_Dataset%20-%20Ecommerce.xlsx%20-%20returns.csv")


In [26]:
customers_df.dtypes

customer_id         str
customer_name       str
segment             str
zip_code          int64
region              str
country             str
city                str
state               str
contact_number      str
joining_date        str
dtype: object

In [27]:
orders_df.dtypes

order_id                                    str
customer_id                                 str
ship_mode                                   str
vendor_id                                   str
order_status                                str
order_purchase_date              datetime64[us]
order_approved_at                           str
order_delivered_carrier_date                str
order_delivered_customer_date               str
order_estimated_delivery_date               str
dtype: object

In [28]:
transactions_df.dtypes

id              int64
order_id          str
product_id        str
sales_amt     float64
qty             int64
discount      float64
profit_amt    float64
dtype: object

In [29]:
returns_df.dtypes

order_id         str
return_reason    str
dtype: object

In [18]:
return_reason_count=returns_df.groupby('return_reason').agg(reason_count=('order_id','count')).reset_index().sort_values('reason_count',ascending=False)
return_reason_count.head(1)

,return_reason,reason_count
1,Not Satisfied,156


In [ ]:
order_metrics[['first_purchase', 'last_purchase']] = order_metrics[['first_purchase', 'last_purchase']].apply(pd.to_datetime)

customer_metrics_df = (
    customers_df[['customer_id', 'customer_name']]
    .merge(order_metrics[['tot_orders', 'first_purchase', 'last_purchase']], left_on='customer_id', right_index=True, how='left')
    .merge(transaction_metrics[['total_order_value', 'total_units']], left_on='customer_id', right_index=True, how='left')
    .merge(return_metrics[['tot_returns']], left_on='customer_id', right_index=True, how='left')
)

customer_metrics_df['tot_returns'] = customer_metrics_df['tot_returns'].fillna(0).astype(int)
customer_metrics_df['avg_basket_size'] = np.floor(customer_metrics_df['total_units'] / customer_metrics_df['tot_orders']).astype('Int64')
customer_metrics_df['avg_basket_value'] = (customer_metrics_df['total_order_value'] / customer_metrics_df['tot_orders']).round(2)
customer_metrics_df['length_of_stay_days'] = (customer_metrics_df['last_purchase'] - customer_metrics_df['first_purchase']).dt.days
customer_metrics_df['order_purchase_frequency'] = np.round(customer_metrics_df['length_of_stay_days'] / customer_metrics_df['tot_orders']).astype('Int64')

customer_metrics_df['frequency_rank'] = customer_metrics_df['order_purchase_frequency'].rank(method='dense', ascending=True)
customer_metrics_df['avg_basket_value_rank'] = customer_metrics_df['avg_basket_value'].rank(method='dense', ascending=False)
customer_metrics_df['avg_rank'] = ((customer_metrics_df['frequency_rank'] + customer_metrics_df['avg_basket_value_rank']) / 2).round(0).astype(int)

customer_metrics_df['Customer_Category'] = pd.cut(
    customer_metrics_df['avg_rank'],
    bins=[-1, 148, 299, float('inf')],
    labels=['Promoters', 'Potentials', 'Detractors']
)

customer_metrics_df['avg_basket_value'] = customer_metrics_df['avg_basket_value'].map('${:,.2f}'.format)

customer_metrics_df = customer_metrics_df[
    ['customer_name', 'tot_orders', 'tot_returns', 'total_order_value', 'avg_basket_size',
     'avg_basket_value', 'length_of_stay_days', 'order_purchase_frequency',
     'avg_rank', 'Customer_Category']
]
customer_metrics_df['avg_basket_value'] = pd.to_numeric(
    customer_metrics_df['avg_basket_value'].str.replace(r'[\$,]', '', regex=True)
)

In [35]:
customer_metrics_df['Customer_Category'].value_counts()

Customer_Category
Potentials    303
Detractors    247
Promoters     243
Name: count, dtype: int64

In [36]:
customer_metrics_df[customer_metrics_df['customer_name']=='Jason Klamczynski']

,customer_name,tot_orders,tot_returns,total_order_value,avg_basket_size,avg_basket_value,length_of_stay_days,order_purchase_frequency,avg_rank,Customer_Category
528,Jason Klamczynski,3,1,18073.84,6,6024.61,289,96,43,Promoters


In [37]:
customer_metrics_df.sort_values('avg_basket_value',ascending=False).head(3)

,customer_name,tot_orders,tot_returns,total_order_value,avg_basket_size,avg_basket_value,length_of_stay_days,order_purchase_frequency,avg_rank,Customer_Category
528,Jason Klamczynski,3,1,18073.840,6,6024.61,289,96,43,Promoters
363,Susan MacKendrick,1,0,4356.686,26,4356.69,0,0,2,Promoters
537,Karen Bern,7,1,26950.968,4,3850.14,385,55,24,Promoters
